In [ ]:
import numpy as np
import pandas as pd

In [ ]:
train = pd.read_csv("./train.csv").drop(columns = "ID")
test = pd.read_csv("./test.csv").drop(columns = "ID")

In [ ]:
train.head()

,설립연도,국가,분야,투자단계,직원 수,인수여부,상장여부,고객수(백만명),총 투자금(억원),연매출(억원),SNS 팔로워 수(백만명),기업가치(백억원),성공확률
0,2009,CT005,이커머스,Series A,4126.0,No,No,56.0,3365.0,4764.0,4.71,NaN,0.3
1,2023,CT006,핀테크,Seed,4167.0,Yes,No,80.0,4069.0,279.0,1.00,2500-3500,0.8
2,2018,CT007,기술,Series A,3132.0,Yes,Yes,54.0,6453.0,12141.0,4.00,3500-4500,0.5
3,2016,CT006,NaN,Seed,3245.0,Yes,Yes,NaN,665.0,10547.0,2.97,NaN,0.7
4,2020,CT002,에듀테크,Seed,1969.0,No,Yes,94.0,829.0,9810.0,1.00,1500-2500,0.1


In [ ]:
test.isnull().sum()

,0
설립연도,0
국가,0
분야,354
투자단계,0
직원 수,76
인수여부,0
상장여부,0
고객수(백만명),547
총 투자금(억원),0
연매출(억원),0


In [ ]:
train.isnull().sum()

,0
설립연도,0
국가,0
분야,857
투자단계,0
직원 수,174
인수여부,0
상장여부,0
고객수(백만명),1320
총 투자금(억원),0
연매출(억원),0


In [ ]:
from sklearn.preprocessing import LabelEncoder
mask = train["분야"] != 0
le = LabelEncoder()
encoded = train["분야"].copy()
encoded[mask] = le.fit_transform(train["분야"][mask])
encoded = encoded.astype(int)
train["분야"] = encoded

In [ ]:
mask = test["분야"] != 0
le = LabelEncoder()
encoded = test["분야"].copy()
encoded[mask] = le.fit_transform(test["분야"][mask])
encoded = encoded.astype(int)
test["분야"] = encoded

In [ ]:
train["분야"].head(10)

,분야
0,6
1,8
2,2
3,10
4,5
5,2
6,10
7,1
8,10
9,10


In [ ]:
train["투자단계"] = le.fit_transform(train['투자단계'])

In [ ]:
test["투자단계"] = le.fit_transform(test['투자단계'])

In [ ]:
train["국가"] = train["국가"].str.extract(r'(\d{2})$').astype(int)

In [ ]:
test["국가"] = test["국가"].str.extract(r'(\d{2})$').astype(int)

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
train_known = train[train['직원 수'].notnull()]
train_unknown = train[train["직원 수"].isnull()]
feature_cols = ["국가", "설립연도", "연매출(억원)"]

X_train = train_known[feature_cols]
y_train = train_known["직원 수"]
X_test = train_unknown[feature_cols]

rdfr = RandomForestRegressor(n_estimators = 100, random_state=42)
rdfr.fit(X_train, y_train)
i_predicted_values = rdfr.predict(X_test)
train.loc[train['직원 수'].isnull(), '직원 수'] = i_predicted_values

In [ ]:
test_unknown = test[test["직원 수"].isnull()]
X_test = test_unknown[feature_cols]

rdfr = RandomForestRegressor(n_estimators = 100, random_state=42)
rdfr.fit(X_train, y_train)
i_predicted_values = rdfr.predict(X_test)
test.loc[test['직원 수'].isnull(), '직원 수'] = i_predicted_values

In [ ]:
train_known = train[train['고객수(백만명)'].notnull()]
train_unknown = train[train["고객수(백만명)"].isnull()]
feature_cols = ["국가", "SNS 팔로워 수(백만명)", "연매출(억원)"]

X_train = train_known[feature_cols]
y_train = train_known["고객수(백만명)"]
X_test = train_unknown[feature_cols]

rdfr = RandomForestRegressor(n_estimators = 100, random_state=42)
rdfr.fit(X_train, y_train)
ii_predicted_values = rdfr.predict(X_test)
train.loc[train['고객수(백만명)'].isnull(), '고객수(백만명)'] = ii_predicted_values

In [ ]:
test_unknown = test[test["고객수(백만명)"].isnull()]
X_test = test_unknown[feature_cols]

rdfr = RandomForestRegressor(n_estimators = 100, random_state=42)
rdfr.fit(X_train, y_train)
ii_predicted_values = rdfr.predict(X_test)
test.loc[test['고객수(백만명)'].isnull(), '고객수(백만명)'] = ii_predicted_values

In [ ]:
def parse_valuation(val):
    if pd.isnull(val):
        return np.nan
    if isinstance(val, str):
        if '-' in val:
            low, high = map(int, val.split('-'))
            return (low + high) / 2
        elif '이상' in val:
            number = int(''.join(filter(str.isdigit, val)))
            return number + 500  # 또는 다른 보정 값 (ex. number * 1.1)
    return np.nan

train["기업가치(백억원)"] = train["기업가치(백억원)"].apply(parse_valuation)
test["기업가치(백억원)"] = test["기업가치(백억원)"].apply(parse_valuation)

In [ ]:
train_known = train[train['기업가치(백억원)'].notnull()]
train_unknown = train[train["기업가치(백억원)"].isnull()]
feature_cols = ["국가", "직원 수", "연매출(억원)", "고객수(백만명)", "분야", "SNS 팔로워 수(백만명)"]

X_train = train_known[feature_cols]
y_train = train_known["기업가치(백억원)"]
X_test = train_unknown[feature_cols]

rdfr = RandomForestRegressor(n_estimators = 100, random_state=42)
rdfr.fit(X_train, y_train)
iii_predicted_values = rdfr.predict(X_test)
train.loc[train['기업가치(백억원)'].isnull(), '기업가치(백억원)'] = iii_predicted_values

In [ ]:
test_unknown = test[test["기업가치(백억원)"].isnull()]
X_test = test_unknown[feature_cols]

rdfr = RandomForestRegressor(n_estimators = 100, random_state=42)
rdfr.fit(X_train, y_train)
iii_predicted_values = rdfr.predict(X_test)
test.loc[test['기업가치(백억원)'].isnull(), '기업가치(백억원)'] = iii_predicted_values

In [ ]:
train["기업가치(백억원)"].head(10)

,기업가치(백억원)
0,4615.0
1,3000.0
2,4000.0
3,4725.0
4,2000.0
5,3000.0
6,3000.0
7,4000.0
8,4295.0
9,4197.5


In [ ]:
train["인수여부"] = train["인수여부"].replace({"Yes": 1, "No": 0})
train["상장여부"] = train["상장여부"].replace({"Yes": 1, "No": 0})

<ipython-input-110-588d4d714ba4>:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  train["인수여부"] = train["인수여부"].replace({"Yes": 1, "No": 0})
<ipython-input-110-588d4d714ba4>:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  train["상장여부"] = train["상장여부"].replace({"Yes": 1, "No": 0})


In [ ]:
test["인수여부"] = test["인수여부"].replace({"Yes": 1, "No": 0})
test["상장여부"] = test["상장여부"].replace({"Yes": 1, "No": 0})

<ipython-input-109-d63ff1e38b81>:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  test["인수여부"] = test["인수여부"].replace({"Yes": 1, "No": 0})
<ipython-input-109-d63ff1e38b81>:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  test["상장여부"] = test["상장여부"].replace({"Yes": 1, "No": 0})


#모델


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

# ▶️ 사용할 피처 선택
#feature_cols = ["연매출(억원)", "설립연도", "직원 수", "고객수(백만명)", "SNS 팔로워 수(백만명)"]

X = train.drop(columns = "성공확률")
y = train["성공확률"]

# ▶️ 학습/검증 데이터 나누기
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# ▶️ 모델 생성 및 학습
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# ▶️ 예측 및 평가
y_pred = model.predict(X_val)
y_pred = np.clip(y_pred, 0, 1)  # 확률값은 0~1로 제한

# ▶️ 평가 지표 출력
print("RMSE:", np.sqrt(mean_squared_error(y_val, y_pred)))
print("R² Score:", r2_score(y_val, y_pred))


RMSE: 0.24133315752787562
R² Score: -0.007724964425565695


In [ ]:
train[["연매출(억원)", "기업가치(백억원)"]].corr()


,연매출(억원),기업가치(백억원)
연매출(억원),1.000000,-0.040655
기업가치(백억원),-0.040655,1.000000


In [ ]:
train.head()

,설립연도,국가,분야,투자단계,직원 수,인수여부,상장여부,고객수(백만명),총 투자금(억원),연매출(억원),SNS 팔로워 수(백만명),기업가치(백억원),성공확률
0,2009,5,6,2,4126.0,0,0,56.00,3365.0,4764.0,4.71,4615.0,0.3
1,2023,6,8,1,4167.0,1,0,80.00,4069.0,279.0,1.00,3000.0,0.8
2,2018,7,2,2,3132.0,1,1,54.00,6453.0,12141.0,4.00,4000.0,0.5
3,2016,6,10,1,3245.0,1,1,29.56,665.0,10547.0,2.97,4725.0,0.7
4,2020,2,5,1,1969.0,0,1,94.00,829.0,9810.0,1.00,2000.0,0.1


In [ ]:
submit = pd.read_csv('./sample_submission.csv')
# Get predictions for the test data
X_test = test  # Assuming 'test' is your preprocessed test DataFrame
y_pred_test = model.predict(X_test)
y_pred_test = np.clip(y_pred_test, 0, 1)  # Clip predictions to 0-1

# Assign predictions to the submission DataFrame
submit['성공확률'] = y_pred_test
submit.to_csv('./submission.csv', encoding='UTF-8-sig', index=False)

In [ ]:
submit = pd.read_csv('./sample_submission.csv')
# 결과 저장
submit['성공확률'] = y_pred
submit.to_csv('./submission.csv', encoding='UTF-8-sig', index=False)

ValueError: Length of values (876) does not match length of index (1755)